# Probability distribution transfer

Monge-Kantorovich and the optimal linear solution

## Theory

<!--## Description-->


<!-- ### The Monge-Kantorovich problem -->
    
Given a probability distribution, how can it be optimally mapped to another distribution.

<!-- http://people.irisa.fr/Nicolas.Courty/OATMIL/images/T.png -->

### References

- [[Olkin1982]](https://www.sciencedirect.com/science/article/pii/0024379582901124)
  I. Olkin, F.Pukelsheim, The distance between two random vectors with given
  dispersion matrices, 1982, https://doi.org/10.1016/0024-3795(82)90112-4

- [[Peyre2019]](https://arxiv.org/abs/1803.00567) G. Peyre, M. Cuturi,
  Computational Optimal Transport, 2019

- [[Flamary2020]](https://arxiv.org/abs/1905.10155) R. Flamary, K. Lounici,
  A. Ferrari, Concentration bounds for linear Monge mapping estimation and
  optimal transport domain adaptation, 2020, arXiv:1905.10155v2
    
- [Villani2003] C. Villani, Topics in Optimal Transportation, 2003


### Notation

* Source probability distribution function $\mu$ of random vector $X$

* Target probability distribution function $\nu$ of random vector $Y$

* Transfer function $T$ transporting $\mu$ to $\nu$ is searched.

In other words, $T^{-1}(Y)=X$, or $T$ pushes distribution
$\mu$ to distribution $\nu$, which is noted as:

$$ T_{\#} \mu= \nu $$

* The random vectors $X$ and $Y$ have dimension $N_{features}$
* The sample size is denoted by $N_{sample}$.




<!-- ### The univariate case -->

For univariate distributions, the transfer map is given by:

$$T(X) = \mathcal{C}_Y^{-1} \circ \mathcal{C}_X \left( X \right)$$

* $\mathcal{C}_X$: cumulative distribution functions of $X$
* $\mathcal{C}_Y$: cumulative distribution functions of $Y$.

### Kantorovich formulation

Following [Villani2003], $d \pi(x, y)$ measures the amount of mass displaced from
$x$ to $y$ where the mass located at $x$ may be splitted to
several locations $y$ and several locations at $y$ may receive mass
from several locations $x$. It is required that:

$$\int_Y d \pi(x, y) = d \mu(x)$$
$$\int_X d \pi(x, y) = d \nu(y)$$
 
The Kantorovich optimal transportation problem is the minimization of:

$$I[\pi] = \int_{X \times Y} c(x, y) d \pi(x, y)$$

Where $c(x, y)$ is the displacement cost from $x$ to $y$ and
is typically chosen as $\| x -y \|^2$.

### Monge formulation

The Monge problem is a special case of the Kantorovich formulation where the
mass from $x$ is not split. Following [Villani2003], the formulation is:

$$ d \pi(x, y) = d \mu(x) \delta(y=T(x)) $$

The transport cost is then:

$$ I[T]=\int_X \| x - T(x) \|^2 d \mu(x) $$

Where we look for the transport map $T$ minimizing $I[T]$. Which can also be looked at in the following way:

$$I[T]=\sum_i \frac{1}{N} \| x_i - T(x_i) \|^2 = E \left[ \| X - T(X) \|^2 \right]$$

### Linear solution

The exact and unique solution for gaussians for both the Monge and the
Kantorovich problem is [[Olkin1982]](https://www.sciencedirect.com/science/article/pii/0024379582901124):

$$ T= \Sigma_u^{ - \frac{1}{2} } \left( \Sigma_u^{ \frac{1}{2} } \Sigma_v \Sigma_u^{ \frac{1}{2} } \right)^{ \frac{1}{2}} \Sigma_u^{ - \frac{1}{2} } $$

* $T$ is the $N_{\text{features}} \times N_{\text{features}}$ linear transfer matrix

* $\Sigma_u$ is the $N_{\text{features}} \times N_{\text{features}}$ covariance matrix of the source

* $\Sigma_v$ is the $N_{\text{features}} \times N_{\text{features}}$ covariance matrix of the target



Given model projection data $P$ of size $N_{\text{variables}}
\times N_{\text{pixels}}$, the output $O$ with PDF transfer is calculated
with:

$$   O = T (P - m_\mu) + m_\nu $$

Where $m_\mu$ and $m_\nu$ are the mean values of the target
(historical) and source (observation) data.

This solution is also the optimal linear transfer function for any
distribution given mean and covariance matrices $\Sigma_u$ and
$\Sigma_v$ [[Flamary2020]](https://arxiv.org/abs/1905.10155).

<!-- Intuitively, the possible linear
transformations consist of a displacement along the mean, and scaling along
variance. With a number of dimensions larger than 1, the further linear
transformation possible is a multidimensional rotation. -->

<!-- ### Linear transformations in 2 dimensions -->

<!-- https://upload.wikimedia.org/wikipedia/commons/2/2c/2D_affine_transformation_matrix.svg -->

## Illustration

### Imports

In [ ]:
import importlib
import numpy as np
import xarray as xr
import pyku
import pyku.analyse as analyse

import scipy
import pyku.pdftransfer as pdftransfer

import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from PIL import Image
import requests
from io import BytesIO

import dask.array as dk

### Images

**Image source**

In [ ]:
img_obs = Image.open(importlib.resources.files('pyku.etc.static') / 'pyku_beach.jpg')
img_cal = Image.open(importlib.resources.files('pyku.etc.static') / 'pyku_offenbach.jpg')

**Function to convert image to netcdf Dataset**

In [ ]:
def img2ds(img):
    
    """
    Convert image to xarray dataset
    
    Arguments:
        img (PIL.Image): Image
        
    Returns:
        xarray.Dataset: Image as a dataset
    """
    
    np_array = np.array(img)
    np_array = np.flipud(np_array)
    
    print(f"Mode: {img.mode}")
    
    da_r = xr.DataArray(
        name='R',
        data=np_array[:,:,0],
        dims=["lat", "lon"],
        coords=dict(
            lat=(["lat",], np.arange(img.size[1])),
            lon=(["lon",], np.arange(img.size[0])),
        ),
        attrs={
            'long_name': 'Red channel',
            'units': '8bits'
        }
    )

    da_g = xr.DataArray(
        name='G',
        data=np_array[:,:,1],
        dims=["lat", "lon"],
        coords=dict(
            lat=(["lat",], np.arange(img.size[1])),
            lon=(["lon",], np.arange(img.size[0])),
        ),
        attrs={
            'long_name': 'Green channel',
            'units': '8bits'
        }
    )

    da_b = xr.DataArray(
        name='B',
        data=np_array[:,:,2],
        dims=["lat", "lon"],
        coords=dict(
            lat=(["lat",], np.arange(img.size[1])),
            lon=(["lon",], np.arange(img.size[0])),
        ),
        attrs={
            'long_name': 'Blue channel',
            'units': '8bits'
        }
    )
   
    return xr.merge([da_r, da_g, da_b])

**Function to convert dataset to an image**

In [ ]:
def ds2img(ds, gamma=None):
    
    """
    Convert xarray Dataset to PIL image
    
    Arguments:
        ds (xarray.Dataset): Dataset
        gamma (float): gamma correction factor
        
    Returns:
        PIL.Image: Dataset as image
    """
    
    import numpy as np
    
    # Get RGB channels
    # ----------------
    
    np_red = ds['R'].values
    np_green = ds['G'].values
    np_blue =ds['B'].values
    
    # Flip image
    # ----------
    
    np_red = np.flipud(np_red)
    np_green = np.flipud(np_green)
    np_blue = np.flipud(np_blue)
    
    # Stack and convert to image
    # --------------------------
    
    np_img = np.stack([np_red, np_green, np_blue], axis=2)
    
    np_img = np_img - np.min(np_img)
    np_img = np_img/np.max(np_img)
    
    # Apply gamma correction
    # ----------------------
    
    if gamma is not None:
        np_img = np_img ** gamma

    np_img = np_img *255
    np_img = np_img.astype(np.uint8)
   
    img = Image.fromarray(np_img)
    
    return img   

**Transformations**

In [ ]:
%%time

min_percentile = 0.001
max_percentile = 0.999

def transform(ds):

    with xr.set_options(keep_attrs=True):

        min_val = min_percentile
        max_val = max_percentile

        R = ds['R']/255
        G = ds['G']/255
        B = ds['B']/255

        R = R.where(dk.logical_or(R < max_val, R.isnull()), other = max_val)
        G = G.where(dk.logical_or(G < max_val, G.isnull()), other = max_val)
        B = B.where(dk.logical_or(B < max_val, B.isnull()), other = max_val)
        
        R = R.where(dk.logical_or(R > min_val, R.isnull()), other = min_val)
        G = G.where(dk.logical_or(G > min_val, G.isnull()), other = min_val)
        B = B.where(dk.logical_or(B > min_val, B.isnull()), other = min_val)
        
        ds = ds.assign(R=scipy.special.logit(R))
        ds = ds.assign(G=scipy.special.logit(G))
        ds = ds.assign(B=scipy.special.logit(B))
        
        return ds
        
def inverse_transform(ds):

    with xr.set_options(keep_attrs=True):
        
        ds = ds.assign(R=scipy.special.expit(ds['R'])*255)
        ds = ds.assign(G=scipy.special.expit(ds['G'])*255)
        ds = ds.assign(B=scipy.special.expit(ds['B'])*255)
        
        return ds    

### Convert images to datasets

In [ ]:
ds_cal = img2ds(img_cal).assign_attrs({'label': 'Source'}).astype('float32')
ds_obs = img2ds(img_obs).assign_attrs({'label': 'Target'}).astype('float32')

### Source, target and pdfs

In [ ]:
plt.ioff()

fig, axs = plt.subplots(2, 2)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Source/Model',
    nbins=32,
    ax=axes[1]
)

axes[2].imshow(img_obs)
axes[2].axis('off')

analyse.pdf(
    ds_obs['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_obs['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_obs['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Target/Observation',
    nbins=32,
    ax=axes[3]
)

plt.gcf().set_size_inches(15, 12)

In [ ]:
plt.show()

### Linear Monge Kantorovich

**Calculations**

In [ ]:
# Initialize corrector
# --------------------

corrector_lmk = pdftransfer.LMKCorrector(
    ds_obs = ds_obs,
    ds_cal = ds_cal
)

In [ ]:
# Fit
# ---

corrector_lmk.fit()

In [ ]:
# Predict
# -------

ds_lmk = corrector_lmk.predict(ds_cal)

In [ ]:
# Assign attributes
# -----------------

ds_lmk.attrs['label'] = 'LMK'

# Convert dataset to image
# ------------------------

img_lmk = ds2img(ds_lmk)

**Visualisation of transformations**

In [ ]:
S = corrector_lmk.method.transfer[0:2,0:2]
display(S)

S = [ [1.03, 0.04],
      [0.04, 1.03]
    ]

display(S)
display(corrector_lmk.method.transfer)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set up a Cartesian grid of points.
XMIN, XMAX, YMIN, YMAX = -3, 3, -3, 3
N = 16
xgrid = np.linspace(XMIN, XMAX, N)
ygrid = np.linspace(YMIN, YMAX, N)
grid = np.array(np.meshgrid(xgrid, ygrid)).reshape(2, N**2)

# Our untransformed unit basis vectors, i and j:
basis = np.array([[2,0], [0,2]])

def plot_quadrilateral(basis, color='k'):
    """Plot the quadrilateral defined by the two basis vectors."""
    ix, iy = basis[0]
    jx, jy = basis[1]
    plt.plot([0, ix, ix+jx, jx, 0], [0, iy, iy+jy, jy, 0], color)

def plot_vector(v, color='k', lw=1):
    """Plot vector v as a line with a specified color and linewidth."""
    plt.plot([0, v[0]], [0, v[1]], c=color, lw=lw)

def plot_points(grid, color='k'):
    """Plot the grid points in a specified color."""
    plt.scatter(*grid, c=color, s=2, alpha=0.5)

def apply_transformation(basis, T):
    """Return the transformed basis after applying transformation T."""
    return (T @ basis.T).T

# The untransformed grid and unit square.
plot_points(grid)
plot_quadrilateral(basis)

# Apply the transformation matrix, S, to the scene.
#S = np.array(((1.5, 0.5),(0.5, 1.5)))
tbasis = apply_transformation(basis, S)
plot_quadrilateral(tbasis, 'r')
tgrid = S @ grid
plot_points(tgrid, 'r')

# Find the eigenvalues and eigenvectors of S... 
vals, vecs = np.linalg.eig(S)
print(vals, vecs)
if all(np.isreal(vals)):
    # ... if they're all real, indicate them on the diagram.
    v1, v2 = vals
    e1, e2 = vecs.T
    plot_vector(v1*e1, 'r', 3)
    plot_vector(v2*e2, 'r', 3)
    plot_vector(e1, 'k')
    plot_vector(e2, 'k')

# Ensure the plot has 1:1 aspect (i.e. squares look square) and set the limits.
plt.axis('square')
plt.xlim(XMIN, XMAX)
plt.ylim(YMIN, YMAX)

plt.show()

**Results**

In [ ]:
plt.ioff()

fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')
axes[0].set_title('Source')

axes[1].imshow(img_obs)
axes[1].axis('off')
axes[1].set_title('Target')

axes[2].imshow(img_lmk)
axes[2].axis('off')
axes[2].set_title('Corrected')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Source',
    nbins=32,
    xlim=[0, None],
    ax=axes[3]
)

analyse.pdf(
    ds_obs['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_obs['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_obs['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Target',
    nbins=32,
    xlim=[0, None],
    ax=axes[4]
)

analyse.pdf(
    ds_lmk['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_lmk['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_lmk['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Corrected',
    nbins=32,
    xlim=[0, None],
    ax=axes[5]
)


plt.gcf().set_size_inches(18, 10)

In [ ]:
plt.show()

### Linear Monge Kantorovich with transformation

**Calculations**

In [ ]:
ds_obs_tr = transform(ds_obs)
ds_cal_tr = transform(ds_cal)

In [ ]:
# Initialize corrector
# --------------------

corrector_lmk_tr = pdftransfer.LMKCorrector(
    ds_obs = ds_obs_tr,
    ds_cal = ds_cal_tr,
)

In [ ]:
# Fit
# ---

corrector_lmk_tr.fit()

In [ ]:
# Predict
# -------

ds_lmk_tr = corrector_lmk_tr.predict(ds_cal_tr)

In [ ]:
ds_lmk_tr = inverse_transform(ds_lmk_tr)

In [ ]:
# Assign attributes
# -----------------

ds_lmk_tr.attrs['label'] = 'LMK transformed'

# Convert dataset to image
# ------------------------

img_lmk_tr = ds2img(ds_lmk_tr)

**Results**

In [ ]:
plt.ioff()

fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')
axes[0].set_title('Source')

axes[1].imshow(img_obs)
axes[1].axis('off')
axes[1].set_title('Target')

axes[2].imshow(img_lmk_tr)
axes[2].axis('off')
axes[2].set_title('Corrected')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Source',
    nbins=32,
    xlim=[0, None],
    ax=axes[3]
)

analyse.pdf(
    ds_obs['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_obs['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_obs['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Target',
    nbins=32,
    xlim=[0, None],
    ax=axes[4]
)

analyse.pdf(
    ds_lmk_tr['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_lmk_tr['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_lmk_tr['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Corrected',
    nbins=32,
    xlim=[0, None],
    ax=axes[5]
)


plt.gcf().set_size_inches(18, 10)

In [ ]:
plt.show()

### Univariate Quantile Mapping (UQM)

**Calculations**

In [ ]:
# Initialize corrector
# --------------------

corrector_uqm = pdftransfer.UQMCorrector(
    ds_obs = ds_obs,
    ds_cal = ds_cal,
    nbins = 1024
)

# Fit
# ---

corrector_uqm.fit()

# Predict
# -------

ds_uqm = corrector_uqm.predict(ds_cal)

# Assign attributes
# -----------------

ds_uqm.attrs['label'] = 'UQM'

img_uqm = ds2img(ds_uqm)

**Results**

In [ ]:
plt.ioff()

fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')
axes[0].set_title('Source')

axes[1].imshow(img_obs)
axes[1].axis('off')
axes[1].set_title('Target')

axes[2].imshow(img_uqm)
axes[2].axis('off')
axes[2].set_title('Corrected')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Source',
    nbins=32,
    ax=axes[3]
)

analyse.pdf(
    ds_obs['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_obs['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_obs['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Target',
    nbins=32,
    ax=axes[4]
)

analyse.pdf(
    ds_uqm['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_uqm['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_uqm['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Corrected',
    nbins=32,
    ax=axes[5]
)


plt.gcf().set_size_inches(18, 10)

In [ ]:
plt.show()

### NDPDF algorithm

**Calculations**

In [ ]:
# Initialize corrector
# --------------------

corrector_ndpdf = pdftransfer.NDPDFCorrector(
    ds_obs = ds_obs,
    ds_cal = ds_cal,
    nbins = 256,
    niterations = 30
)

# Fit
# ---

corrector_ndpdf.fit()

# Predict
# -------

ds_ndpdf = corrector_ndpdf.predict(ds_cal)

# Assign attributes
# -----------------

ds_ndpdf.attrs['label'] = 'NDPDF'

# Convert dataset to image
# ------------------------

img_ndpdf = ds2img(ds_ndpdf)

**Results**

In [ ]:
fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')
axes[0].set_title('Source')

axes[1].imshow(img_obs)
axes[1].axis('off')
axes[1].set_title('Target')

axes[2].imshow(img_lmk)
axes[2].axis('off')
axes[2].set_title('Corrected')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Source',
    nbins=32,
    ax=axes[3]
)

analyse.pdf(
    ds_obs['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_obs['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_obs['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Target',
    nbins=32,
    ax=axes[4]
)

analyse.pdf(
    ds_ndpdf['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_ndpdf['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_ndpdf['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Corrected',
    nbins=32,
    ax=axes[5]
)


plt.gcf().set_size_inches(18, 10)

In [ ]:
plt.show()

### Method comparison (pdfs)

In [ ]:
fig, axs = plt.subplots(5, 2)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')

analyse.pdf(
    ds_cal['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_cal['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_cal['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='Original',
    nbins=32,
    ax=axes[1]
)

axes[2].imshow(img_lmk)
axes[2].axis('off')

analyse.pdf(
    ds_lmk['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_lmk['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_lmk['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='LMK',
    nbins=32,
    ax=axes[3]
)

axes[4].imshow(img_lmk_tr)
axes[4].axis('off')

analyse.pdf(
    ds_lmk_tr['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_lmk_tr['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_lmk_tr['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='LMK',
    nbins=32,
    ax=axes[5]
)

axes[6].imshow(img_uqm)
axes[6].axis('off')

analyse.pdf(
    ds_uqm['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_uqm['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_uqm['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='UQM',
    nbins=32,
    ax=axes[7]
)

axes[8].imshow(img_ndpdf)
axes[8].axis('off')

analyse.pdf(
    ds_ndpdf['R'].assign_attrs({'label': 'Red', 'long_name': 'channels'}),
    ds_ndpdf['G'].assign_attrs({'label': 'Green', 'long_name': 'channels'}),
    ds_ndpdf['B'].assign_attrs({'label': 'Blue', 'long_name': 'channels'}),
    title='NDPDF',
    nbins=32,
    ax=axes[9]
)

plt.gcf().set_size_inches(15, 28)

In [ ]:
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

axes[0].imshow(img_cal)
axes[0].axis('off')
axes[0].set_title('Source')

axes[1].imshow(img_obs)
axes[1].axis('off')
axes[1].set_title('Target')

axes[2].imshow(img_uqm)
axes[2].axis('off')
axes[2].set_title('UQM')

axes[3].imshow(img_ndpdf)
axes[3].axis('off')
axes[3].set_title('NDPDF')

axes[4].imshow(img_lmk)
axes[4].axis('off')
axes[4].set_title('LMK')

axes[5].imshow(img_lmk_tr)
axes[5].axis('off')
axes[5].set_title('LMK+transform')

plt.gcf().set_size_inches(15, 7.5)

fig.tight_layout()

### Method comparison (images)

In [ ]:
plt.show()

### Method comparison (density plots and correlation)



| Method    | Marginale  | Correlations | Applicable outside training range| Transport | Comment                         |
|-----------|------------|--------------|----------------------------------|-----------|---------------------------------|
| UQM       | Yes        | no           | No                               | map       |                                 |
| NDPDF     | Yes        | yes          | No                               | plan      |                                 |
| LMK       | Approx     | yes          | Yes                              | map       |                                 |
| QDM       | Yes        | no           | Yes, *detrending*                | plan      |                                 |
| MBCn      | Yes        | yes          | Yes, *detrending*                | plan      |                                 |


In [ ]:
fig, axs = plt.subplots(2, 3)

axes=axs.reshape(-1)

sample_size = 4000

analyse.pdf2D(
    ds_obs,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[0]
)

analyse.pdf2D(
    ds_cal,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[1]
)

analyse.pdf2D(
    ds_uqm,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[2]
)

analyse.pdf2D(
    ds_lmk,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[3]
)

analyse.pdf2D(
    ds_ndpdf,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[4]
)

analyse.pdf2D(
    ds_lmk_tr,
    varx='R',
    vary='B',
    sample_size=sample_size,
    xlim=(-50, 300),
    ylim=(-50, 250),
    ax=axes[5]
)

plt.gcf().set_size_inches(15, 10)
fig.tight_layout()

In [ ]:
plt.show()